# Analytical Groundwater Modeling — Exercises 1.1 to 1.3
**Chapter 1: Steady One-Dimensional Flow**

This notebook solves Exercises 1.1, 1.2, and 1.3 from *Analytical Groundwater Modeling* (Mark Bakker & Vincent Post, 2022).

---

## Background

We consider **areal recharge between two parallel rivers** that fully penetrate an unconfined aquifer of constant transmissivity $T = kH$.

### Governing equations

The **head solution** (Eq. 1.24):
$$h = -\frac{N}{2T}(x^2 - Lx) + \frac{(h_L - h_0)x}{L} + h_0$$

The **discharge vector** (Eq. 1.25):
$$Q_x = N\left(x - \frac{L}{2}\right) - T\frac{h_L - h_0}{L}$$

Boundary conditions:
- $h|_{x=0} = h_0$
- $h|_{x=L} = h_L$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── shared style ──────────────────────────────────────────────
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

---
## Base Parameters (shared by all exercises)

In [ ]:
# ── aquifer parameters ────────────────────────────────────────
L  = 1000   # aquifer length [m]
H  = 10     # saturated thickness [m]
zb = -5     # aquifer bottom elevation [m]
k  = 10     # hydraulic conductivity [m/d]
n  = 0.3    # porosity [-]
T  = k * H  # transmissivity [m²/d]  →  100 m²/d

# ── boundary heads (original example) ────────────────────────
h0 = 6      # head at left  river (x = 0)  [m]
hL = 4      # head at right river (x = L)  [m]
N  = 0.001  # areal recharge [m/d]

x  = np.linspace(0, L, 500)   # spatial grid

print(f'Transmissivity T = {T} m²/d')
print(f'Total recharge  N·L = {N*L} m²/d')

---

## Exercise 1.1 — Location of the Water Divide

### Problem statement
> *Compute the location of the water divide for the parameters provided above.
> Verify that the location of the water divide is consistent with the discharge figures computed above.*

### Theory

The **water divide** is the location $x^*$ where the horizontal hydraulic gradient equals zero, i.e. $Q_x = 0$.  
Setting Eq. 1.25 to zero:

$$0 = N\left(x^* - \frac{L}{2}\right) - T\frac{h_L - h_0}{L}$$

Solving for $x^*$:

$$\boxed{x^* = \frac{L}{2} + \frac{T(h_L - h_0)}{NL}}$$

### Verification logic

All recharge falling between $x = 0$ and $x = x^*$ drains **left** into the left river,  
and all recharge between $x = x^*$ and $x = L$ drains **right** into the right river:

$$Q_{\text{left}} = N \cdot x^* \qquad Q_{\text{right}} = N \cdot (L - x^*)$$

These must match $|Q_x(0)|$ and $Q_x(L)$ from the analytical discharge expression.

In [ ]:
# ── Analytical solution (original parameters) ─────────────────
h_ex1  = -N / (2*T) * (x**2 - L*x) + (hL - h0) * x / L + h0
Qx_ex1 = N * (x - L/2) - T * (hL - h0) / L

# ── Water divide location (analytical) ───────────────────────
x_divide = L/2 + T*(hL - h0) / (N * L)
print(f'Water divide location  x* = {x_divide:.1f} m')

# ── Discharge into each river (analytical formula) ────────────
Qleft_formula  = -Qx_ex1[0]         # positive value = flow INTO left river
Qright_formula =  Qx_ex1[-1]        # positive value = flow INTO right river
print(f'\nDischarge into left  river  (from formula): {Qleft_formula:.4f} m²/d')
print(f'Discharge into right river  (from formula): {Qright_formula:.4f} m²/d')

# ── Verification via water divide location ────────────────────
Qleft_verify  = N * x_divide          # recharge in [0, x*]
Qright_verify = N * (L - x_divide)    # recharge in [x*, L]
print(f'\nVerification via x*:')
print(f'  N·x*        = {Qleft_verify:.4f} m²/d  → should equal Qleft')
print(f'  N·(L - x*)  = {Qright_verify:.4f} m²/d  → should equal Qright')
print(f'  Total check N·L = {Qleft_verify + Qright_verify:.4f} m²/d  (should be {N*L:.4f} m²/d)')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# ── Head plot ─────────────────────────────────────────────────
ax1.plot(x, h_ex1, 'steelblue', lw=2, label='h(x)')
ax1.axvline(x_divide, color='firebrick', ls='--', lw=1.5, label=f'water divide x* = {x_divide:.0f} m')
ax1.scatter([x_divide], [np.interp(x_divide, x, h_ex1)],
            color='firebrick', zorder=5, s=60)
ax1.set_xlabel('x (m)'); ax1.set_ylabel('head (m)')
ax1.set_title('Exercise 1.1 — Head'); ax1.legend(); ax1.grid(alpha=0.3)

# ── Discharge plot ────────────────────────────────────────────
ax2.plot(x, Qx_ex1, 'darkorange', lw=2, label='Qx(x)')
ax2.axhline(0, color='k', lw=0.8)
ax2.axvline(x_divide, color='firebrick', ls='--', lw=1.5, label=f'Qx = 0 at x* = {x_divide:.0f} m')
ax2.scatter([x_divide], [0], color='firebrick', zorder=5, s=60)
ax2.annotate(f'Qleft = {Qleft_formula:.3f}', xy=(0, Qx_ex1[0]),
             xytext=(80, Qx_ex1[0] - 0.08), fontsize=9,
             arrowprops=dict(arrowstyle='->', color='k'))
ax2.annotate(f'Qright = {Qright_formula:.3f}', xy=(L, Qx_ex1[-1]),
             xytext=(750, Qx_ex1[-1] + 0.04), fontsize=9,
             arrowprops=dict(arrowstyle='->', color='k'))
ax2.set_xlabel('x (m)'); ax2.set_ylabel('Qx (m²/d)')
ax2.set_title('Exercise 1.1 — Discharge'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ex1_1.png', bbox_inches='tight')
plt.show()
print('Figure saved.')

### Exercise 1.1 — Summary

| Quantity | Value |
|---|---|
| Water divide location $x^*$ | **300 m** |
| Discharge to left river $N \cdot x^*$ | **0.300 m²/d** |
| Discharge to right river $N \cdot (L - x^*)$ | **0.700 m²/d** |
| Sum (= total recharge $NL$) | **1.000 m²/d** ✓ |

The analytical formula gives $x^* = L/2 + T(h_L - h_0)/(NL) = 500 + 100 \cdot (-2)/1 = 300$ m.  
The divide is shifted **left** of centre because the right river head is lower, pulling more water rightward.

---

## Exercise 1.2 — Left River Head for Zero Left-to-Right Flow

### Problem statement
> *Determine what the water level in the left river must be so that exactly all water
> that infiltrates flows to the right river, while no water from the left river flows to
> the right river. Plot h and Qx vs. x.*

### Theory

The condition **"all recharge flows right, none flows left"** means:

- The water divide sits exactly at $x^* = 0$ (the left boundary).
- $Q_x(0) = 0$: the discharge at $x = 0$ is zero — no flow enters or leaves through the left river.

Setting $Q_x = 0$ at $x = 0$ in Eq. 1.25:

$$0 = N\left(0 - \frac{L}{2}\right) - T\frac{h_L - h_0}{L}$$

$$\frac{T(h_0 - h_L)}{L} = \frac{NL}{2}$$

$$\boxed{h_0 = h_L + \frac{NL^2}{2T}}$$

In [ ]:
# ── Required h0 for divide at x = 0 ──────────────────────────
h0_ex2 = hL + N * L**2 / (2 * T)
print(f'Required left river head  h0 = {h0_ex2:.4f} m')
print(f'(Original h0 was {h0} m; the left river must be HIGHER to push all water right)')

# ── Solution ──────────────────────────────────────────────────
h_ex2  = -N / (2*T) * (x**2 - L*x) + (hL - h0_ex2) * x / L + h0_ex2
Qx_ex2 = N * (x - L/2) - T * (hL - h0_ex2) / L

# ── Verify ────────────────────────────────────────────────────
print(f'\nVerification:')
print(f'  Qx at x = 0  : {Qx_ex2[0]:.6f} m²/d  (should be ≈ 0)')
print(f'  Qx at x = L  : {Qx_ex2[-1]:.4f} m²/d  (should equal N·L = {N*L:.4f} m²/d)')
print(f'  Water divide  : x* = L/2 + T(hL-h0)/(NL) = {L/2 + T*(hL - h0_ex2)/(N*L):.1f} m')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# ── Head ──────────────────────────────────────────────────────
ax1.plot(x, h_ex2, 'steelblue', lw=2)
ax1.scatter([0, L], [h0_ex2, hL], color=['steelblue','steelblue'], zorder=5, s=60)
ax1.annotate(f'h₀ = {h0_ex2:.1f} m', (0, h0_ex2), xytext=(40, h0_ex2 - 0.15), fontsize=9)
ax1.annotate(f'hL = {hL} m', (L, hL), xytext=(820, hL + 0.1), fontsize=9)
ax1.set_xlabel('x (m)'); ax1.set_ylabel('head (m)')
ax1.set_title('Exercise 1.2 — Head'); ax1.grid(alpha=0.3)

# ── Discharge ─────────────────────────────────────────────────
ax2.plot(x, Qx_ex2, 'darkorange', lw=2)
ax2.axhline(0, color='k', lw=0.8)
ax2.scatter([0], [Qx_ex2[0]], color='firebrick', zorder=5, s=60,
            label=f'Qx(0) = {Qx_ex2[0]:.3f} m²/d')
ax2.scatter([L], [Qx_ex2[-1]], color='green', zorder=5, s=60,
            label=f'Qx(L) = {Qx_ex2[-1]:.3f} m²/d')
ax2.set_xlabel('x (m)'); ax2.set_ylabel('Qx (m²/d)')
ax2.set_title('Exercise 1.2 — Discharge')
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ex1_2.png', bbox_inches='tight')
plt.show()
print('Figure saved.')

### Exercise 1.2 — Summary

The required left-river head is:

$$h_0 = h_L + \frac{NL^2}{2T} = 4 + \frac{0.001 \times 1000^2}{2 \times 100} = 4 + 5 = \mathbf{9\text{ m}}$$

**Interpretation:** With $h_0 = 9$ m, the head gradient from the left river is exactly steep enough  
that the leftward pressure from the high water table is balanced by the rightward pull from the  
lower right river. The water divide sits at $x = 0$, so $Q_x(0) = 0$: the left river neither  
supplies water to nor receives water from the aquifer. All recharge ($NL = 1$ m²/d) exits through  
the right river.

---

## Exercise 1.3 — Infer Areal Recharge from an Interior Head Measurement

### Problem statement
> *Consider the case that $h_0 = h_L = 10$ m. The head halfway between the two rivers  
> is measured to be 9 m. Compute the areal recharge for this case using the parameters  
> of the example above. Plot h vs. x.*

### Theory

With **equal** river heads ($h_0 = h_L = \hat{h}$), the linear gradient term vanishes:

$$h = -\frac{N}{2T}(x^2 - Lx) + \hat{h}$$

At the midpoint $x = L/2$, $(x^2 - Lx) = \frac{L^2}{4} - \frac{L^2}{2} = -\frac{L^2}{4}$, so:

$$h\!\left(\frac{L}{2}\right) = \frac{N L^2}{8T} + \hat{h}$$

Solving for $N$:

$$\boxed{N = \frac{8T}{L^2}\left[h\!\left(\frac{L}{2}\right) - \hat{h}\right]}$$

**Physical note:** If $h(L/2) < \hat{h}$, then $N < 0$, which represents **net evapotranspiration**  
(groundwater is being extracted from the aquifer by plants/evaporation faster than it recharges).

In [1]:
# ── Exercise 1.3 parameters ───────────────────────────────────
h0_ex3  = 10.0   # head at left  river [m]
hL_ex3  = 10.0   # head at right river [m]  (equal rivers)
h_mid   =  9.0   # measured head at x = L/2 [m]

# ── Infer N ───────────────────────────────────────────────────
N_ex3 = 8 * T / L**2 * (h_mid - h0_ex3)
print(f'Inferred areal recharge  N = {N_ex3:.6f} m/d')
print(f'                           = {N_ex3*1000:.4f} mm/d')
print(f'                           = {N_ex3*365*1000:.1f} mm/yr')

if N_ex3 < 0:
    print('\n⚠  N < 0 → net evapotranspiration (the aquifer is losing water, not gaining it).')
else:
    print('\nN > 0 → net infiltration / recharge.')

# ── Solution ──────────────────────────────────────────────────
h_ex3  = -N_ex3 / (2*T) * (x**2 - L*x) + (hL_ex3 - h0_ex3) * x / L + h0_ex3
Qx_ex3 = N_ex3 * (x - L/2) - T * (hL_ex3 - h0_ex3) / L

# ── Verification ──────────────────────────────────────────────
h_mid_check = np.interp(L/2, x, h_ex3)
print(f'\nVerification: h(L/2) = {h_mid_check:.6f} m  (should be {h_mid} m) ✓')

NameError: name 'T' is not defined

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# ── Head ──────────────────────────────────────────────────────
ax1.plot(x, h_ex3, 'steelblue', lw=2, label='h(x)')
ax1.scatter([0, L/2, L], [h0_ex3, h_mid, hL_ex3],
            color=['steelblue', 'firebrick', 'steelblue'],
            zorder=5, s=80,
            label=['River heads', 'Measured midpoint', ''])
ax1.annotate(f'h(L/2) = {h_mid} m\n(measured)', xy=(L/2, h_mid),
             xytext=(550, 9.2), fontsize=9, color='firebrick',
             arrowprops=dict(arrowstyle='->', color='firebrick'))
ax1.annotate(f'h₀ = hL = {h0_ex3:.0f} m', xy=(0, h0_ex3),
             xytext=(50, 9.85), fontsize=9)
ax1.set_xlabel('x (m)'); ax1.set_ylabel('head (m)')
ax1.set_title(f'Exercise 1.3 — Head  (N = {N_ex3:.5f} m/d)')
ax1.set_ylim(8.8, 10.2); ax1.grid(alpha=0.3)

# ── Discharge ─────────────────────────────────────────────────
ax2.plot(x, Qx_ex3, 'darkorange', lw=2, label='Qx(x)')
ax2.axhline(0, color='k', lw=0.8)
ax2.scatter([L/2], [0], color='firebrick', zorder=5, s=60,
            label=f'Qx = 0 at x = L/2\n(divide, symmetric case)')
ax2.set_xlabel('x (m)'); ax2.set_ylabel('Qx (m²/d)')
ax2.set_title('Exercise 1.3 — Discharge')
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ex1_3.png', bbox_inches='tight')
plt.show()
print('Figure saved.')

### Exercise 1.3 — Summary

$$N = \frac{8T}{L^2}\left(h_{\text{mid}} - \hat{h}\right) = \frac{8 \times 100}{1{,}000{,}000}(9 - 10) = -0.0008 \text{ m/d} \approx -292 \text{ mm/yr}$$

**The negative sign indicates net evapotranspiration**, not recharge.  
The aquifer loses water to the atmosphere faster than precipitation replenishes it.  
Because both river heads are equal, the flow field is **symmetric** about $x = L/2$:  
half the extracted groundwater is supplied by the left river, half by the right.

---

## Comparison of All Three Cases

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
cases = [
    ('Ex 1.1\n(original, N=+0.001)', x, h_ex1, Qx_ex1, x_divide),
    ('Ex 1.2\n(h₀=9 m, all flow right)', x, h_ex2, Qx_ex2, 0.0),
    ('Ex 1.3\n(h₀=hL=10 m, N=−0.0008)', x, h_ex3, Qx_ex3, L/2),
]

for i, (title, xi, hi, Qi, xd) in enumerate(cases):
    ax_h  = axes[0, i]
    ax_q  = axes[1, i]

    ax_h.plot(xi, hi, 'steelblue', lw=2)
    ax_h.axvline(xd, color='firebrick', ls='--', lw=1.2, label=f'x*={xd:.0f} m')
    ax_h.set_title(title, fontsize=10); ax_h.set_ylabel('h (m)'); ax_h.grid(alpha=0.3)
    ax_h.legend(fontsize=8)

    ax_q.plot(xi, Qi, 'darkorange', lw=2)
    ax_q.axhline(0, color='k', lw=0.8)
    ax_q.axvline(xd, color='firebrick', ls='--', lw=1.2)
    ax_q.set_ylabel('Qx (m²/d)'); ax_q.set_xlabel('x (m)'); ax_q.grid(alpha=0.3)

plt.suptitle('Comparison: Exercises 1.1 – 1.3', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ex1_comparison.png', bbox_inches='tight')
plt.show()

---
## Key Equations Reference

| Quantity | Formula |
|---|---|
| Head | $h = -\dfrac{N}{2T}(x^2 - Lx) + \dfrac{(h_L-h_0)x}{L} + h_0$ |
| Discharge | $Q_x = N\!\left(x - \dfrac{L}{2}\right) - T\dfrac{h_L - h_0}{L}$ |
| Water divide | $x^* = \dfrac{L}{2} + \dfrac{T(h_L - h_0)}{NL}$ |
| $h_0$ for zero left outflow | $h_0 = h_L + \dfrac{NL^2}{2T}$ |
| Infer $N$ from $h(L/2)$ | $N = \dfrac{8T}{L^2}\!\left[h(L/2) - h_0\right]$ (when $h_0 = h_L$) |